In [1]:
import pandas as pd
import numpy as np

project_path = r"C:\Users\Bawan Singh\OneDrive\Documents\Arshleen\market-portfolio-risk-analysis"

In [2]:
portfolio_returns = pd.read_csv(
    project_path + r"\data\processed\portfolio_returns.csv",
    index_col=0,
    parse_dates=True
)

In [3]:
portfolio_returns.head()

,Conservative,Balanced,Growth
Date,,,
2019-01-03,-0.003180,-0.009997,-0.024742
2019-01-04,0.007151,0.015816,0.033633
2019-01-07,0.002666,0.004192,0.008812
2019-01-08,0.002165,0.004578,0.008018
2019-01-09,0.002527,0.002977,0.005788


In [4]:
confidence_level = 0.95

In [5]:
historical_var = portfolio_returns.quantile(1 - confidence_level)

In [6]:
historical_var

Conservative   -0.010521
Balanced       -0.011563
Growth         -0.018348
Name: 0.050000000000000044, dtype: float64

In [7]:
expected_shortfall = {}

for portfolio in portfolio_returns.columns:
    var = historical_var[portfolio]
    
    tail_losses = portfolio_returns[portfolio][
        portfolio_returns[portfolio] <= var
    ]
    
    expected_shortfall[portfolio] = tail_losses.mean()

expected_shortfall = pd.Series(expected_shortfall)

In [8]:
expected_shortfall

Conservative   -0.015438
Balanced       -0.018586
Growth         -0.029343
dtype: float64

In [9]:
risk_summary = pd.DataFrame({
    "Historical VaR (95%)": historical_var,
    "Expected Shortfall (95%)": expected_shortfall
})


In [10]:
risk_summary

,Historical VaR (95%),Expected Shortfall (95%)
Conservative,-0.010521,-0.015438
Balanced,-0.011563,-0.018586
Growth,-0.018348,-0.029343


In [11]:
rolling_volatility = portfolio_returns.rolling(window=30).std()

In [12]:
rolling_volatility.head(35)

,Conservative,Balanced,Growth
Date,,,
2019-01-03,NaN,NaN,NaN
2019-01-04,NaN,NaN,NaN
2019-01-07,NaN,NaN,NaN
2019-01-08,NaN,NaN,NaN
2019-01-09,NaN,NaN,NaN
2019-01-10,NaN,NaN,NaN
2019-01-11,NaN,NaN,NaN
2019-01-14,NaN,NaN,NaN
2019-01-15,NaN,NaN,NaN


In [13]:
rolling_volatility_annualized = rolling_volatility * np.sqrt(252)

In [14]:
rolling_volatility_annualized.head(35)

,Conservative,Balanced,Growth
Date,,,
2019-01-03,NaN,NaN,NaN
2019-01-04,NaN,NaN,NaN
2019-01-07,NaN,NaN,NaN
2019-01-08,NaN,NaN,NaN
2019-01-09,NaN,NaN,NaN
2019-01-10,NaN,NaN,NaN
2019-01-11,NaN,NaN,NaN
2019-01-14,NaN,NaN,NaN
2019-01-15,NaN,NaN,NaN


In [15]:
rolling_volatility_annualized.max()

Conservative    0.323070
Balanced        0.449790
Growth          0.707038
dtype: float64

In [16]:
rolling_volatility_annualized.idxmax()

Conservative   2020-04-06
Balanced       2020-04-08
Growth         2020-04-08
dtype: datetime64[ns]

In [18]:
cumulative_returns = (1 + portfolio_returns).cumprod()

running_peak = cumulative_returns.cummax()

drawdown = (cumulative_returns - running_peak) / running_peak

In [19]:
drawdown.idxmin()

Conservative   2022-10-20
Balanced       2022-10-14
Growth         2022-10-14
dtype: datetime64[ns]

In [20]:
drawdown.min()

Conservative   -0.250688
Balanced       -0.248956
Growth         -0.303071
dtype: float64

In [21]:
id="q4k8nm"
drawdown.idxmin()

Conservative   2022-10-20
Balanced       2022-10-14
Growth         2022-10-14
dtype: datetime64[ns]

In [22]:
id="v6p2xr"
drawdown.head()

,Conservative,Balanced,Growth
Date,,,
2019-01-03,0.0,0.0,0.0
2019-01-04,0.0,0.0,0.0
2019-01-07,0.0,0.0,0.0
2019-01-08,0.0,0.0,0.0
2019-01-09,0.0,0.0,0.0


In [23]:
covid_period = portfolio_returns.loc["2020-02-01":"2020-06-30"]

covid_period_return = (1 + covid_period).prod() - 1

covid_period_return

Conservative    0.084777
Balanced        0.052294
Growth          0.075082
dtype: float64

In [24]:
stress_2022 = portfolio_returns.loc["2022-01-01":"2022-12-31"]

stress_2022_return = (1 + stress_2022).prod() - 1

stress_2022_return

Conservative   -0.197331
Balanced       -0.199534
Growth         -0.265425
dtype: float64

In [25]:
risk_summary = pd.DataFrame({
    "Historical VaR (95%)": historical_var,
    "Expected Shortfall (95%)": expected_shortfall,
    "Maximum Drawdown": drawdown.min(),
    "Maximum Rolling Volatility": rolling_volatility_annualized.max()
})

risk_summary

,Historical VaR (95%),Expected Shortfall (95%),Maximum Drawdown,Maximum Rolling Volatility
Conservative,-0.010521,-0.015438,-0.250688,0.323070
Balanced,-0.011563,-0.018586,-0.248956,0.449790
Growth,-0.018348,-0.029343,-0.303071,0.707038


In [26]:
risk_summary.to_csv(
    project_path + r"\data\processed\risk_summary.csv"
)

In [27]:
risk_summary

,Historical VaR (95%),Expected Shortfall (95%),Maximum Drawdown,Maximum Rolling Volatility
Conservative,-0.010521,-0.015438,-0.250688,0.323070
Balanced,-0.011563,-0.018586,-0.248956,0.449790
Growth,-0.018348,-0.029343,-0.303071,0.707038


In [28]:
stress_summary = pd.DataFrame({
    "2020 Stress Period Return": covid_period_return,
    "2022 Annual Return": stress_2022_return
})

stress_summary

,2020 Stress Period Return,2022 Annual Return
Conservative,0.084777,-0.197331
Balanced,0.052294,-0.199534
Growth,0.075082,-0.265425


In [29]:
stress_summary.to_csv(
    project_path + r"\data\processed\stress_summary.csv"
)